# Canada Vapes Marketing Revenue Growth and Customer Targeting

## Goal

Turn the cleaned transaction data into practical marketing decisions that can increase repeat purchases, average order value, customer retention, and net sales.

The notebook creates:

- customer-level behavioural segments;
- campaign-ready customer target lists;
- retention, coupon, channel, product, and cross-sell insights;
- revenue-concentration and customer-value analysis;
- adjustable revenue-opportunity scenarios; and
- presentation-ready marketing visuals.

## Important profitability limitation

The dataset contains revenue and net sales but does not contain product cost, cost of goods sold, marketing spend, shipping cost, or gross margin. Therefore:

- the notebook can identify customers and campaigns that are likely to generate more **net sales**;
- it uses a clearly labelled **profitability proxy score** based on customer value, frequency, full-price purchasing, and low refund behaviour; but
- it cannot calculate true profit, contribution margin, customer acquisition cost, or return on advertising spend.

Add cost and marketing-spend data before making final profit-allocation decisions.

## Setup and execution order

The analysis is divided into separate steps and code cells. Run the cells from top to bottom.

In VS Code, select **Restart Kernel**, then select **Run All**. Do not begin with the `purchase_rows` cell because the earlier cells create `df`, the completed-purchase flags, and the order-aggregation method.

The dataset path is `C:/Users/tooko/Downloads/canadavape/cleaned_sales_data.csv`.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing_packages]
    )
else:
    print("All required packages are available.")

In [ ]:
import itertools
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import FuncFormatter, PercentFormatter

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

BLUE = "#2F5D8C"
GOLD = "#C79A3B"
ORANGE = "#C96A3D"
OLIVE = "#7A8450"
PINK = "#B85C7A"
INK = "#2C333A"
GREY = "#A7AFB7"
LIGHT_GREY = "#E7EBEF"

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": INK,
        "axes.labelcolor": INK,
        "text.color": INK,
        "xtick.color": INK,
        "ytick.color": INK,
        "axes.titleweight": "bold",
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "legend.frameon": False,
    }
)

currency_formatter = FuncFormatter(lambda value, _: f"${value:,.0f}")
integer_formatter = FuncFormatter(lambda value, _: f"{value:,.0f}")

### 1. Set file locations and campaign assumptions

In [ ]:
# Exact location of the cleaned Canada Vapes dataset
DATA_FILE = Path(
    "C:/Users/tooko/Downloads/canadavape/cleaned_sales_data.csv"
)

# Stop with a clear message if the file cannot be found
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE}\n"
        "Confirm that cleaned_sales_data.csv is in the canadavape folder."
    )

OUTPUT_FOLDER = DATA_FILE.parent
VISUALS_FOLDER = OUTPUT_FOLDER / "canada_vapes_marketing_visuals"
TARGETS_FOLDER = OUTPUT_FOLDER / "canada_vapes_campaign_targets"
VISUALS_FOLDER.mkdir(parents=True, exist_ok=True)
TARGETS_FOLDER.mkdir(parents=True, exist_ok=True)

CUSTOMER_FILE = OUTPUT_FOLDER / "marketing_customer_segments.csv"
CAMPAIGN_SUMMARY_FILE = OUTPUT_FOLDER / "marketing_campaign_summary.csv"
INSIGHT_FILE = OUTPUT_FOLDER / "marketing_insight_summary.csv"

# Adjustable scenario assumptions. These are not guaranteed forecasts.
REACTIVATION_RATE = 0.10
SECOND_PURCHASE_CONVERSION_RATE = 0.10
CROSS_SELL_AOV_LIFT = 0.05
VIP_AOV_LIFT = 0.03
COUPON_REDUCTION_RATE = 0.10

# Cohort and chart settings
COHORT_MONTHS = 12
MIN_COHORT_CUSTOMERS = 30
MAX_COHORT_ROWS = 24
RANDOM_STATE = 42

print("Data file:", DATA_FILE)
print("Visuals folder:", VISUALS_FOLDER)
print("Campaign target folder:", TARGETS_FOLDER)

### 2. Load and validate the cleaned data

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned file not found: {DATA_FILE}\n"
        "Run the cleaning notebook first or correct DATA_FILE."
    )

df = pd.read_csv(DATA_FILE, low_memory=False)

required_columns = [
    "date", "order_id", "revenue", "status", "customer_id",
    "customer_type", "products", "items_sold", "coupon",
    "net_sales", "attribution",
]
missing_columns = [
    column for column in required_columns if column not in df.columns
]
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
for column in ["revenue", "net_sales", "items_sold"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

if "coupon_used" in df.columns:
    if df["coupon_used"].dtype != bool:
        df["coupon_used"] = (
            df["coupon_used"]
              .astype("string")
              .str.lower()
              .isin(["true", "1", "yes", "y"])
        )
else:
    coupon_text = df["coupon"].astype("string").str.strip().str.lower()
    no_coupon = {"", "no", "none", "0", "not used", "no coupon"}
    df["coupon_used"] = coupon_text.notna() & ~coupon_text.isin(no_coupon)

for column in ["status", "customer_type", "products", "attribution"]:
    df[column] = df[column].astype("string").fillna("Unknown")

print(f"Rows: {len(df):,}")
print(f"Unique order numbers: {df['order_id'].nunique(dropna=True):,}")
print(f"Identifiable customers: {df['customer_id'].nunique(dropna=True):,}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
display(df.head())

### 3. Define completed purchases and validate the order grain

Positive purchases with Completed, Curbside-Complete, or Home-Delivery-Com status are used for customer-value and retention analysis. Refunds, fraud, reshipments, and incomplete orders do not count as new purchases.

In [ ]:
status_text = df["status"].astype("string")
completed_statuses = [
    "Completed", "Curbside-Complete", "Home-Delivery-Com"
]

df["is_completed_purchase"] = (
    status_text.isin(completed_statuses)
    & df["net_sales"].gt(0).fillna(False)
)
df["is_refund_or_negative"] = (
    status_text.str.contains("refund|return", case=False, na=False)
    | df["net_sales"].lt(0).fillna(False)
)

df["product_group"] = df["products"].astype("string").str.strip()
df.loc[
    df["product_group"].str.lower().isin(["", "unknown", "nan", "none"]),
    "product_group",
] = "Unknown"

duplicate_order_rows = df["order_id"].duplicated(keep=False) & df["order_id"].notna()
duplicate_order_rate = duplicate_order_rows.mean()

if duplicate_order_rows.any():
    duplicate_orders = df.loc[duplicate_order_rows]
    repeated_total_share = (
        duplicate_orders.groupby("order_id")["net_sales"]
        .nunique(dropna=False)
        .eq(1)
        .mean()
    )
else:
    repeated_total_share = 0.0

# If duplicate order rows mostly repeat the same order total, use max.
# Otherwise, treat rows as additive product lines and use sum.
order_value_method = "max" if repeated_total_share >= 0.80 else "sum"

print(f"Rows attached to duplicate order numbers: {duplicate_order_rate:.2%}")
print(f"Duplicate orders with repeated net-sales totals: {repeated_total_share:.2%}")
print("Order-value aggregation method:", order_value_method)

### 4. Build one row per completed customer order

In [ ]:
if "df" not in globals() or "is_completed_purchase" not in df.columns:
    raise RuntimeError(
        "Required setup is missing. In VS Code, select Restart Kernel and then Run All."
    )

purchase_rows = df.loc[
    df["is_completed_purchase"]
    & df["customer_id"].notna()
    & df["order_id"].notna()
    & df["date"].notna()
].copy()

if purchase_rows.empty:
    raise ValueError("No identifiable completed purchases were found.")

def mode_or_first(series):
    non_missing = series.dropna()
    if non_missing.empty:
        return "Unknown"
    modes = non_missing.mode()
    return modes.iloc[0] if not modes.empty else non_missing.iloc[0]

def join_unique_products(series):
    values = [
        str(value).strip()
        for value in series.dropna()
        if str(value).strip() and str(value).strip().lower() != "unknown"
    ]
    return " | ".join(sorted(set(values))) if values else "Unknown"

aggregation_map = {
    "order_date": ("date", "max"),
    "net_sales": ("net_sales", order_value_method),
    "revenue": ("revenue", order_value_method),
    "items_sold": ("items_sold", order_value_method),
    "coupon_used": ("coupon_used", "max"),
    "customer_type": ("customer_type", mode_or_first),
    "attribution": ("attribution", mode_or_first),
    "products": ("product_group", join_unique_products),
}

order_level = (
    purchase_rows.groupby(["customer_id", "order_id"], as_index=False)
    .agg(**aggregation_map)
)

print(f"Completed customer orders: {len(order_level):,}")
display(order_level.head())

### 5. Build customer marketing features

In [ ]:
analysis_date = order_level["order_date"].max() + pd.Timedelta(days=1)

customer_features = (
    order_level.groupby("customer_id")
    .agg(
        first_purchase=("order_date", "min"),
        last_purchase=("order_date", "max"),
        frequency=("order_id", "nunique"),
        monetary_value=("net_sales", "sum"),
        average_order_value=("net_sales", "mean"),
        total_items=("items_sold", "sum"),
        average_items_per_order=("items_sold", "mean"),
        coupon_usage_rate=("coupon_used", "mean"),
        primary_attribution=("attribution", mode_or_first),
        initial_customer_type=("customer_type", mode_or_first),
    )
    .reset_index()
)

customer_features["recency_days"] = (
    analysis_date - customer_features["last_purchase"]
).dt.days
customer_features["tenure_days"] = (
    customer_features["last_purchase"]
    - customer_features["first_purchase"]
).dt.days
customer_features["full_price_order_rate"] = (
    1 - customer_features["coupon_usage_rate"]
)

known_product_variety = (
    purchase_rows.loc[purchase_rows["product_group"].ne("Unknown")]
    .groupby("customer_id")["product_group"]
    .nunique()
)
customer_features["product_variety"] = (
    customer_features["customer_id"]
    .map(known_product_variety)
    .fillna(0)
    .astype(int)
)

refund_metrics = (
    df.loc[df["customer_id"].notna()]
    .groupby("customer_id")
    .agg(
        all_records=("order_id", "size"),
        refund_records=("is_refund_or_negative", "sum"),
    )
)
refund_metrics["refund_rate"] = (
    refund_metrics["refund_records"]
    / refund_metrics["all_records"].replace(0, np.nan)
)
customer_features["refund_rate"] = (
    customer_features["customer_id"]
    .map(refund_metrics["refund_rate"])
    .fillna(0)
)

# Average days between purchases
order_intervals = order_level.sort_values(["customer_id", "order_date"]).copy()
order_intervals["days_since_previous_order"] = (
    order_intervals.groupby("customer_id")["order_date"].diff().dt.days
)
average_intervals = order_intervals.groupby("customer_id")[
    "days_since_previous_order"
].mean()
customer_features["average_days_between_orders"] = (
    customer_features["customer_id"].map(average_intervals)
)

display(customer_features.head())

### 6. Score customer value, behaviour, and profitability potential

In [ ]:
def percentile_score(series, higher_is_better=True):
    score = series.rank(method="average", pct=True).fillna(0)
    return score if higher_is_better else 1 - score

def quintile_score(series, higher_is_better=True):
    percentile = series.rank(method="average", pct=True)
    score = np.ceil(percentile * 5).clip(1, 5).astype(int)
    return score if higher_is_better else 6 - score

customer_features["recency_score"] = quintile_score(
    customer_features["recency_days"], higher_is_better=False
)
customer_features["frequency_score"] = quintile_score(
    customer_features["frequency"], higher_is_better=True
)
customer_features["monetary_score"] = quintile_score(
    customer_features["monetary_value"], higher_is_better=True
)
customer_features["aov_score"] = quintile_score(
    customer_features["average_order_value"], higher_is_better=True
)

monetary_percentile = percentile_score(customer_features["monetary_value"])
frequency_percentile = percentile_score(customer_features["frequency"])
aov_percentile = percentile_score(customer_features["average_order_value"])
full_price_percentile = percentile_score(customer_features["full_price_order_rate"])
low_refund_percentile = percentile_score(
    customer_features["refund_rate"], higher_is_better=False
)

# This is a prioritization score, not accounting profit.
customer_features["profitability_proxy_score"] = 100 * (
    0.35 * monetary_percentile
    + 0.20 * frequency_percentile
    + 0.15 * aov_percentile
    + 0.15 * full_price_percentile
    + 0.15 * low_refund_percentile
)

customer_features["rfm_total"] = customer_features[[
    "recency_score", "frequency_score", "monetary_score"
]].sum(axis=1)

### 7. Assign actionable marketing segments

In [ ]:
segment_conditions = [
    customer_features["recency_score"].ge(4)
    & customer_features["frequency_score"].ge(4)
    & customer_features["monetary_score"].ge(4),

    customer_features["recency_score"].le(2)
    & customer_features["monetary_score"].ge(4),

    customer_features["frequency"].eq(1)
    & customer_features["recency_score"].ge(4)
    & customer_features["aov_score"].ge(4),

    customer_features["frequency_score"].ge(4)
    & customer_features["monetary_score"].ge(3),

    customer_features["coupon_usage_rate"].ge(0.75)
    & customer_features["frequency"].ge(2),

    customer_features["frequency_score"].ge(4)
    & customer_features["aov_score"].le(2),

    customer_features["frequency"].eq(1)
    & customer_features["recency_score"].ge(3),

    customer_features["recency_score"].le(2)
    & customer_features["frequency_score"].le(2),
]

segment_names = [
    "Champions",
    "At-Risk High Value",
    "New High Potential",
    "Loyal High Value",
    "Coupon Driven",
    "Frequent Low AOV",
    "Recent One-Time Buyer",
    "Hibernating",
]

customer_features["marketing_segment"] = np.select(
    segment_conditions,
    segment_names,
    default="Developing",
)

customer_features.to_csv(CUSTOMER_FILE, index=False)

segment_summary = (
    customer_features.groupby("marketing_segment")
    .agg(
        customers=("customer_id", "nunique"),
        historical_net_sales=("monetary_value", "sum"),
        median_customer_value=("monetary_value", "median"),
        median_order_value=("average_order_value", "median"),
        median_frequency=("frequency", "median"),
        median_recency_days=("recency_days", "median"),
        average_coupon_rate=("coupon_usage_rate", "mean"),
        average_refund_rate=("refund_rate", "mean"),
        average_profitability_proxy=("profitability_proxy_score", "mean"),
    )
    .reset_index()
)

display(segment_summary.sort_values("historical_net_sales", ascending=False))
print("Customer file saved:", CUSTOMER_FILE)

### 8. Create campaign recommendations and target files

In [ ]:
campaign_rules = {
    "Champions": {
        "objective": "Protect and expand high-value relationships",
        "offer": "VIP access, new-product previews, referral rewards, and premium bundles",
        "primary_kpi": "Repeat rate and incremental average order value",
    },
    "At-Risk High Value": {
        "objective": "Reactivate valuable customers before they lapse",
        "offer": "Personalized replenishment reminder or limited-time comeback bundle",
        "primary_kpi": "Reactivation rate and recovered net sales",
    },
    "New High Potential": {
        "objective": "Convert a strong first purchase into a second purchase",
        "offer": "Product education, replenishment timing, and a relevant second-order recommendation",
        "primary_kpi": "Second-purchase conversion within 30 to 60 days",
    },
    "Loyal High Value": {
        "objective": "Increase retention and product breadth",
        "offer": "Loyalty benefits, subscriptions, and personalized cross-sell bundles",
        "primary_kpi": "Retention, product variety, and customer value",
    },
    "Coupon Driven": {
        "objective": "Reduce unnecessary discount dependence",
        "offer": "Use minimum-spend thresholds, bundles, and loyalty points instead of broad discounts",
        "primary_kpi": "Full-price order rate and net sales after discount",
    },
    "Frequent Low AOV": {
        "objective": "Increase basket size",
        "offer": "Complementary bundles, multi-buy offers, and free-shipping thresholds",
        "primary_kpi": "Average order value and items per order",
    },
    "Recent One-Time Buyer": {
        "objective": "Generate the second purchase",
        "offer": "Welcome sequence, usage education, and next-best-product recommendation",
        "primary_kpi": "Second-purchase conversion",
    },
    "Hibernating": {
        "objective": "Win back customers selectively",
        "offer": "Low-cost email reactivation; suppress after repeated non-response",
        "primary_kpi": "Reactivation value minus campaign cost",
    },
    "Developing": {
        "objective": "Build purchase frequency and learn preferences",
        "offer": "Personalized content and light product recommendations",
        "primary_kpi": "Purchase frequency and product engagement",
    },
}

campaign_summary = segment_summary.copy()
campaign_summary["objective"] = campaign_summary["marketing_segment"].map(
    lambda segment: campaign_rules[segment]["objective"]
)
campaign_summary["recommended_offer"] = campaign_summary["marketing_segment"].map(
    lambda segment: campaign_rules[segment]["offer"]
)
campaign_summary["primary_kpi"] = campaign_summary["marketing_segment"].map(
    lambda segment: campaign_rules[segment]["primary_kpi"]
)
campaign_summary.to_csv(CAMPAIGN_SUMMARY_FILE, index=False)

file_names = {
    "Champions": "target_champions.csv",
    "At-Risk High Value": "target_at_risk_high_value.csv",
    "New High Potential": "target_new_high_potential.csv",
    "Loyal High Value": "target_loyal_high_value.csv",
    "Coupon Driven": "target_coupon_driven.csv",
    "Frequent Low AOV": "target_frequent_low_aov.csv",
    "Recent One-Time Buyer": "target_recent_one_time_buyers.csv",
    "Hibernating": "target_hibernating.csv",
    "Developing": "target_developing.csv",
}

target_columns = [
    "customer_id", "marketing_segment", "last_purchase", "recency_days",
    "frequency", "monetary_value", "average_order_value",
    "average_items_per_order", "coupon_usage_rate", "refund_rate",
    "product_variety", "primary_attribution", "profitability_proxy_score",
]

for segment, file_name in file_names.items():
    target_data = (
        customer_features.loc[
            customer_features["marketing_segment"].eq(segment),
            target_columns,
        ]
        .sort_values(
            ["profitability_proxy_score", "monetary_value"],
            ascending=False,
        )
    )
    target_data.to_csv(TARGETS_FOLDER / file_name, index=False)

print("Campaign summary saved:", CAMPAIGN_SUMMARY_FILE)
print("Target files saved in:", TARGETS_FOLDER)
display(campaign_summary)

### 9. Chart helper functions

In [ ]:
def save_figure(fig, file_name):
    output_path = VISUALS_FOLDER / file_name
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    print("Saved:", output_path)

def label_vertical_bars(ax, format_type="number"):
    for container in ax.containers:
        if format_type == "currency":
            labels = [f"${value:,.0f}" for value in container.datavalues]
        elif format_type == "percent":
            labels = [f"{value:.1f}%" for value in container.datavalues]
        else:
            labels = [f"{value:,.0f}" for value in container.datavalues]
        ax.bar_label(container, labels=labels, padding=3, fontsize=9)

## Marketing Insight 1 — Marketing KPI Scorecard

**Decision:** Establish the size of the customer base, repeat-purchase behaviour, customer identification coverage, and revenue opportunity.

In [ ]:
identifiable_customer_rate = df["customer_id"].notna().mean()
repeat_customer_rate = customer_features["frequency"].gt(1).mean()
total_positive_net_sales = order_level["net_sales"].sum()
median_customer_value = customer_features["monetary_value"].median()
median_order_value = order_level["net_sales"].median()
coupon_order_rate = order_level["coupon_used"].mean()

kpi_values = [
    ("Completed customer net sales", f"${total_positive_net_sales:,.0f}"),
    ("Identifiable customers", f"{len(customer_features):,}"),
    ("Repeat-customer rate", f"{repeat_customer_rate:.1%}"),
    ("Median customer value", f"${median_customer_value:,.2f}"),
    ("Median order value", f"${median_order_value:,.2f}"),
    ("Coupon order rate", f"{coupon_order_rate:.1%}"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, (label, value) in zip(axes.flat, kpi_values):
    ax.axis("off")
    ax.text(0.5, 0.62, value, ha="center", va="center", fontsize=25, weight="bold", color=BLUE)
    ax.text(0.5, 0.30, label, ha="center", va="center", fontsize=11, color=INK, wrap=True)
    ax.add_patch(
        plt.Rectangle((0.02, 0.08), 0.96, 0.84, fill=False, edgecolor=LIGHT_GREY, linewidth=1.5, transform=ax.transAxes)
    )
fig.suptitle("Marketing Customer and Revenue Scorecard", fontsize=16, weight="bold")
save_figure(fig, "01_marketing_kpi_scorecard.png")

## Marketing Insight 2 — New Versus Returning Customer Revenue

**Decision:** Determine whether growth is being driven by acquisition or by returning customers.

In [ ]:
monthly_type_sales = (
    order_level.assign(
        month=lambda data: data["order_date"].dt.to_period("M").dt.to_timestamp()
    )
    .groupby(["month", "customer_type"])["net_sales"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)

type_order = [
    column for column in ["New", "Returning", "Unknown"]
    if column in monthly_type_sales.columns
]
type_order += [column for column in monthly_type_sales.columns if column not in type_order]
monthly_type_sales = monthly_type_sales[type_order]
palette = [GOLD, BLUE, GREY, ORANGE][:len(monthly_type_sales.columns)]

fig, ax = plt.subplots(figsize=(13, 6))
ax.stackplot(
    monthly_type_sales.index,
    *[monthly_type_sales[column] for column in monthly_type_sales.columns],
    labels=monthly_type_sales.columns,
    colors=palette,
    alpha=0.9,
)
ax.set_title("Monthly Net Sales by Customer Type")
ax.set_xlabel("Month")
ax.set_ylabel("Net sales")
ax.yaxis.set_major_formatter(currency_formatter)
ax.legend(loc="upper left", ncol=min(3, len(monthly_type_sales.columns)))
ax.grid(axis="x", visible=False)
save_figure(fig, "02_monthly_sales_by_customer_type.png")

## Marketing Insight 3 — Customer Revenue Concentration

**Decision:** Determine how dependent the company is on a small group of customers and identify the customers who should receive retention protection.

In [ ]:
pareto = customer_features.sort_values("monetary_value", ascending=False).copy()
pareto["customer_rank"] = np.arange(1, len(pareto) + 1)
pareto["customer_share"] = pareto["customer_rank"] / len(pareto)
pareto["cumulative_sales_share"] = (
    pareto["monetary_value"].cumsum() / pareto["monetary_value"].sum()
)

customer_share_for_80_percent = pareto.loc[
    pareto["cumulative_sales_share"].ge(0.80), "customer_share"
].iloc[0]

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(
    pareto["customer_share"], pareto["cumulative_sales_share"],
    color=BLUE, linewidth=2.5,
)
ax.axhline(0.80, color=GOLD, linestyle="--", linewidth=1.8, label="80% of customer sales")
ax.axvline(customer_share_for_80_percent, color=ORANGE, linestyle="--", linewidth=1.8)
ax.scatter([customer_share_for_80_percent], [0.80], color=ORANGE, s=60, zorder=3)
ax.text(
    customer_share_for_80_percent, 0.73,
    f"{customer_share_for_80_percent:.1%} of customers",
    ha="center", fontsize=10,
)
ax.set_title("Customer Revenue Concentration")
ax.set_xlabel("Cumulative share of identifiable customers")
ax.set_ylabel("Cumulative share of completed customer net sales")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.legend()
save_figure(fig, "03_customer_revenue_concentration.png")

print(f"{customer_share_for_80_percent:.1%} of identifiable customers generate 80% of completed customer net sales.")

## Marketing Insight 4 — Segment Size and Revenue Contribution

**Decision:** Prioritize segments using both customer volume and historical value.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

size_plot = segment_summary.sort_values("customers")
axes[0].barh(size_plot["marketing_segment"], size_plot["customers"], color=BLUE)
axes[0].set_title("Customers by Marketing Segment")
axes[0].set_xlabel("Customers")
axes[0].set_ylabel("")
axes[0].xaxis.set_major_formatter(integer_formatter)
axes[0].grid(axis="y", visible=False)

sales_plot = segment_summary.sort_values("historical_net_sales")
axes[1].barh(sales_plot["marketing_segment"], sales_plot["historical_net_sales"], color=GOLD)
axes[1].set_title("Historical Net Sales by Marketing Segment")
axes[1].set_xlabel("Completed customer net sales")
axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(currency_formatter)
axes[1].grid(axis="y", visible=False)

save_figure(fig, "04_segment_size_and_revenue.png")

## Marketing Insight 5 — Customer Frequency Versus Value

**Decision:** Separate high-value loyal customers from one-time, low-value, and developing customers.

In [ ]:
scatter_data = customer_features.loc[
    customer_features["frequency"].gt(0)
    & customer_features["monetary_value"].gt(0)
].copy()
scatter_data = scatter_data.sample(
    n=min(7000, len(scatter_data)), random_state=RANDOM_STATE
)

segment_order = sorted(scatter_data["marketing_segment"].unique())
segment_palette = dict(
    zip(segment_order, sns.color_palette("colorblind", len(segment_order)))
)

fig, ax = plt.subplots(figsize=(11, 8))
for segment in segment_order:
    segment_data = scatter_data.loc[
        scatter_data["marketing_segment"].eq(segment)
    ]
    ax.scatter(
        segment_data["frequency"], segment_data["monetary_value"],
        s=28, alpha=0.45, color=segment_palette[segment],
        label=segment, edgecolors="none",
    )
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Customer Purchase Frequency Versus Historical Value")
ax.set_xlabel("Unique completed orders — logarithmic scale")
ax.set_ylabel("Completed customer net sales — logarithmic scale")
ax.yaxis.set_major_formatter(currency_formatter)
ax.legend(title="Marketing segment", bbox_to_anchor=(1.02, 1), loc="upper left")
save_figure(fig, "05_customer_frequency_vs_value.png")

## Marketing Insight 6 — At-Risk High-Value Customers

**Decision:** Identify valuable customers whose historical spending is high but whose latest purchase is old.

In [ ]:
risk_sample = customer_features.loc[
    customer_features["monetary_value"].gt(0)
    & customer_features["recency_days"].gt(0)
].sample(
    n=min(7000, len(customer_features)), random_state=RANDOM_STATE
)

value_threshold = customer_features["monetary_value"].quantile(0.80)
recency_threshold = customer_features["recency_days"].quantile(0.60)

fig, ax = plt.subplots(figsize=(11, 7))
highlight = risk_sample["marketing_segment"].eq("At-Risk High Value")
ax.scatter(
    risk_sample.loc[~highlight, "recency_days"],
    risk_sample.loc[~highlight, "monetary_value"],
    color=GREY, alpha=0.25, s=18, label="Other customers", edgecolors="none",
)
ax.scatter(
    risk_sample.loc[highlight, "recency_days"],
    risk_sample.loc[highlight, "monetary_value"],
    color=ORANGE, alpha=0.75, s=35, label="At-Risk High Value", edgecolors="none",
)
ax.axhline(value_threshold, color=INK, linestyle="--", linewidth=1.2, label="Top-20% value threshold")
ax.axvline(recency_threshold, color=GOLD, linestyle="--", linewidth=1.2, label="Recency threshold")
ax.set_yscale("log")
ax.set_title("Customer Recency Versus Historical Value")
ax.set_xlabel("Days since latest completed purchase")
ax.set_ylabel("Historical completed net sales — logarithmic scale")
ax.yaxis.set_major_formatter(currency_formatter)
ax.legend(loc="best")
save_figure(fig, "06_at_risk_high_value_customers.png")

## Marketing Insight 7 — Customer-Cohort Retention

**Decision:** Determine whether newly acquired customers return in later months.

In [ ]:
cohort_orders = order_level[["customer_id", "order_date"]].drop_duplicates().copy()
cohort_orders["order_month"] = cohort_orders["order_date"].dt.to_period("M").dt.to_timestamp()
first_month = cohort_orders.groupby("customer_id")["order_month"].min()
cohort_orders["cohort_month"] = cohort_orders["customer_id"].map(first_month)
cohort_orders["cohort_index"] = (
    (cohort_orders["order_month"].dt.year - cohort_orders["cohort_month"].dt.year) * 12
    + cohort_orders["order_month"].dt.month
    - cohort_orders["cohort_month"].dt.month
)

cohort_counts = (
    cohort_orders.groupby(["cohort_month", "cohort_index"])["customer_id"]
    .nunique()
    .unstack(fill_value=0)
)
cohort_sizes = cohort_counts.get(0, pd.Series(index=cohort_counts.index, dtype=float))
eligible_cohorts = cohort_sizes.loc[cohort_sizes.ge(MIN_COHORT_CUSTOMERS)].index

if len(eligible_cohorts) == 0:
    eligible_cohorts = cohort_sizes.loc[cohort_sizes.gt(0)].index
    print("No cohorts met MIN_COHORT_CUSTOMERS; displaying all non-empty cohorts.")

if len(eligible_cohorts) > MAX_COHORT_ROWS:
    eligible_cohorts = eligible_cohorts[-MAX_COHORT_ROWS:]
    print(f"Displaying the most recent {MAX_COHORT_ROWS} eligible cohorts for readability.")

retention = cohort_counts.loc[eligible_cohorts].div(
    cohort_sizes.loc[eligible_cohorts], axis=0
)
retention = retention.loc[:, [column for column in retention.columns if column <= COHORT_MONTHS]]
retention.index = retention.index.strftime("%Y-%m")

fig_height = max(6, min(11, 0.40 * len(retention)))
fig, ax = plt.subplots(figsize=(13, fig_height))
sns.heatmap(
    retention,
    cmap=sns.light_palette(BLUE, as_cmap=True),
    vmin=0,
    vmax=1,
    annot=len(retention) <= 24,
    fmt=".0%",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Customer retention rate"},
    ax=ax,
)
ax.set_title("Monthly Customer-Cohort Retention")
ax.set_xlabel("Months since first completed purchase")
ax.set_ylabel("First-purchase cohort")
save_figure(fig, "07_customer_cohort_retention.png")

## Marketing Insight 8 — Time to the Second Purchase

**Decision:** Choose when to send replenishment, education, and second-purchase messages.

In [ ]:
ordered_dates = (
    order_level.sort_values(["customer_id", "order_date"])
    .groupby("customer_id")["order_date"]
    .apply(list)
)
days_to_second_purchase = ordered_dates.map(
    lambda dates: (dates[1] - dates[0]).days if len(dates) >= 2 else np.nan
).dropna()
days_to_second_purchase = days_to_second_purchase.loc[days_to_second_purchase.ge(0)]

if days_to_second_purchase.empty:
    print("No customers with at least two completed purchases were found.")
else:
    upper_limit = days_to_second_purchase.quantile(0.95)
    display_intervals = days_to_second_purchase.clip(upper=upper_limit)
    median_second_purchase = days_to_second_purchase.median()

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.hist(display_intervals, bins=35, color=BLUE, edgecolor="white", alpha=0.9)
    ax.axvline(
        median_second_purchase, color=GOLD, linestyle="--", linewidth=2,
        label=f"Median: {median_second_purchase:.0f} days",
    )
    ax.set_title("Days from First to Second Completed Purchase")
    ax.set_xlabel("Days; displayed through the 95th percentile")
    ax.set_ylabel("Customers")
    ax.yaxis.set_major_formatter(integer_formatter)
    ax.legend()
    ax.grid(axis="x", visible=False)
    save_figure(fig, "08_time_to_second_purchase.png")
    print(f"Median time to second purchase: {median_second_purchase:.0f} days")

## Marketing Insight 9 — Average Order Value by Segment

**Decision:** Identify segments where bundling, cross-selling, premium products, or minimum-spend thresholds may increase basket value.

In [ ]:
segment_aov = (
    customer_features.groupby("marketing_segment")
    .agg(
        median_aov=("average_order_value", "median"),
        customers=("customer_id", "nunique"),
    )
    .sort_values("median_aov")
)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(segment_aov.index, segment_aov["median_aov"], color=BLUE)
ax.set_title("Median Customer Average Order Value by Segment")
ax.set_xlabel("Median customer average order value")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(currency_formatter)
ax.grid(axis="y", visible=False)
for bar in bars:
    ax.text(
        bar.get_width(), bar.get_y() + bar.get_height() / 2,
        f" ${bar.get_width():,.2f}", va="center", fontsize=9,
    )
save_figure(fig, "09_aov_by_marketing_segment.png")
display(segment_aov.sort_values("median_aov", ascending=False))

## Marketing Insight 10 — Coupon Use by Segment

**Decision:** Determine which customers respond to coupons and where broad discounts may be unnecessary.

This is an observational comparison. It does not prove that coupons caused higher or lower spending.

In [ ]:
order_segments = order_level.merge(
    customer_features[["customer_id", "marketing_segment"]],
    on="customer_id",
    how="left",
)

coupon_segment = (
    order_segments.groupby(["marketing_segment", "coupon_used"])
    .agg(
        orders=("order_id", "nunique"),
        median_order_value=("net_sales", "median"),
    )
    .reset_index()
)
coupon_pivot = coupon_segment.pivot(
    index="marketing_segment",
    columns="coupon_used",
    values="median_order_value",
).rename(columns={False: "No coupon", True: "Coupon used"})

available_columns = [column for column in ["No coupon", "Coupon used"] if column in coupon_pivot.columns]
coupon_pivot = coupon_pivot[available_columns]
coupon_pivot = coupon_pivot.sort_values(
    available_columns[-1] if available_columns else coupon_pivot.columns[0]
)

fig, ax = plt.subplots(figsize=(13, 7))
coupon_pivot.plot(
    kind="barh", ax=ax,
    color=[BLUE, GOLD][:len(coupon_pivot.columns)],
)
ax.set_title("Median Order Value by Segment and Coupon Use")
ax.set_xlabel("Median completed-order net sales")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(currency_formatter)
ax.legend(title="Coupon use")
ax.grid(axis="y", visible=False)
save_figure(fig, "10_coupon_value_by_segment.png")
display(coupon_pivot)

## Marketing Insight 11 — Attribution Source Quality

**Decision:** Identify channels that bring valuable, repeat customers—not only high order volume.

Marketing cost is unavailable, so this is not ROAS. It compares customer value and repeat behaviour by the customer's most common attribution source.

In [ ]:
channel_quality = (
    customer_features.groupby("primary_attribution")
    .agg(
        customers=("customer_id", "nunique"),
        customer_net_sales=("monetary_value", "sum"),
        median_customer_value=("monetary_value", "median"),
        repeat_customer_rate=("frequency", lambda values: values.gt(1).mean()),
        average_profitability_proxy=("profitability_proxy_score", "mean"),
    )
    .reset_index()
)
channel_quality = channel_quality.loc[channel_quality["customers"].ge(10)].copy()
channel_quality["net_sales_per_customer"] = (
    channel_quality["customer_net_sales"] / channel_quality["customers"]
)

top_channels = channel_quality.nlargest(12, "customer_net_sales").copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
value_plot = top_channels.sort_values("net_sales_per_customer")
axes[0].barh(
    value_plot["primary_attribution"], value_plot["net_sales_per_customer"], color=BLUE
)
axes[0].set_title("Net Sales per Customer by Attribution")
axes[0].set_xlabel("Completed net sales per identifiable customer")
axes[0].set_ylabel("")
axes[0].xaxis.set_major_formatter(currency_formatter)
axes[0].grid(axis="y", visible=False)

repeat_plot = top_channels.sort_values("repeat_customer_rate")
axes[1].barh(
    repeat_plot["primary_attribution"], repeat_plot["repeat_customer_rate"] * 100, color=GOLD
)
axes[1].set_title("Repeat-Customer Rate by Attribution")
axes[1].set_xlabel("Customers with more than one completed order")
axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(PercentFormatter(100))
axes[1].grid(axis="y", visible=False)

save_figure(fig, "11_attribution_customer_quality.png")
display(channel_quality.sort_values("customer_net_sales", ascending=False).head(15))

## Marketing Insight 12 — Product Revenue and Customer Reach

**Decision:** Identify products that drive both revenue and broad customer demand.

In [ ]:
known_product_rows = purchase_rows.loc[
    purchase_rows["product_group"].ne("Unknown")
].copy()

product_performance = (
    known_product_rows.groupby("product_group")
    .agg(
        net_sales=("net_sales", "sum"),
        customers=("customer_id", "nunique"),
        orders=("order_id", "nunique"),
        median_order_value=("net_sales", "median"),
    )
    .reset_index()
)

top_products = product_performance.nlargest(15, "net_sales").copy()

fig, axes = plt.subplots(1, 2, figsize=(17, 8))
revenue_plot = top_products.sort_values("net_sales")
axes[0].barh(revenue_plot["product_group"], revenue_plot["net_sales"], color=BLUE)
axes[0].set_title("Top Known Product Descriptions by Net Sales")
axes[0].set_xlabel("Completed net sales")
axes[0].set_ylabel("")
axes[0].xaxis.set_major_formatter(currency_formatter)
axes[0].grid(axis="y", visible=False)

reach_plot = top_products.sort_values("customers")
axes[1].barh(reach_plot["product_group"], reach_plot["customers"], color=GOLD)
axes[1].set_title("Customer Reach for the Same Products")
axes[1].set_xlabel("Identifiable customers")
axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(integer_formatter)
axes[1].grid(axis="y", visible=False)

save_figure(fig, "12_product_revenue_and_customer_reach.png")
display(top_products.sort_values("net_sales", ascending=False))

## Marketing Insight 13 — Cross-Sell Product Pairs

**Decision:** Identify products that frequently appear together and may support bundles or personalized recommendations.

The code splits combined product text using pipes, semicolons, commas, plus signs, and line breaks. Review the resulting product names because commas within a product name can create incorrect splits.

In [ ]:
def split_products(product_text):
    if pd.isna(product_text) or str(product_text).strip().lower() == "unknown":
        return []
    parts = re.split(r"\s*(?:\||;|,|\+|\n)\s*", str(product_text))
    return sorted({part.strip() for part in parts if part.strip()})

order_product_lists = order_level[["order_id", "products"]].copy()
order_product_lists["product_list"] = order_product_lists["products"].map(split_products)

pair_records = []
for products in order_product_lists["product_list"]:
    if len(products) >= 2:
        pair_records.extend(itertools.combinations(products, 2))

if pair_records:
    product_pairs = (
        pd.DataFrame(pair_records, columns=["product_a", "product_b"])
        .value_counts()
        .rename("orders_together")
        .reset_index()
    )
    top_pairs = product_pairs.head(15).copy()
    top_pairs["pair"] = top_pairs["product_a"] + " + " + top_pairs["product_b"]
    top_pairs = top_pairs.sort_values("orders_together")

    fig, ax = plt.subplots(figsize=(13, 8))
    ax.barh(top_pairs["pair"], top_pairs["orders_together"], color=OLIVE)
    ax.set_title("Most Frequent Product Pairs")
    ax.set_xlabel("Completed orders containing both products")
    ax.set_ylabel("")
    ax.xaxis.set_major_formatter(integer_formatter)
    ax.grid(axis="y", visible=False)
    save_figure(fig, "13_cross_sell_product_pairs.png")
    display(top_pairs.sort_values("orders_together", ascending=False))
else:
    product_pairs = pd.DataFrame(columns=["product_a", "product_b", "orders_together"])
    print(
        "No multi-product combinations could be parsed. "
        "Use a product-line dataset if each current row contains only one combined product description."
    )

## Marketing Insight 14 — Profitability Proxy by Segment

**Decision:** Prioritize groups that combine customer value, purchase frequency, higher full-price purchasing, and lower refund behaviour.

This is a marketing prioritization score, not accounting profit.

In [ ]:
proxy_by_segment = (
    customer_features.groupby("marketing_segment")
    .agg(
        customers=("customer_id", "nunique"),
        profitability_proxy=("profitability_proxy_score", "mean"),
        full_price_rate=("full_price_order_rate", "mean"),
        refund_rate=("refund_rate", "mean"),
    )
    .sort_values("profitability_proxy")
)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(proxy_by_segment.index, proxy_by_segment["profitability_proxy"], color=BLUE)
ax.set_title("Marketing Profitability Proxy by Segment")
ax.set_xlabel("Profitability proxy score from 0 to 100")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.grid(axis="y", visible=False)
for bar in bars:
    ax.text(
        bar.get_width(), bar.get_y() + bar.get_height() / 2,
        f" {bar.get_width():.1f}", va="center", fontsize=9,
    )
save_figure(fig, "14_profitability_proxy_by_segment.png")
display(proxy_by_segment.sort_values("profitability_proxy", ascending=False))

## Marketing Insight 15 — Revenue Opportunity Scenarios

**Decision:** Compare the potential scale of several campaign opportunities using adjustable assumptions.

These are planning scenarios, not forecasts. They assume that only a specified percentage of targeted customers responds and use observed customer or order values as the spending benchmark.

In [ ]:
segment_lookup = customer_features.set_index("marketing_segment")

at_risk = customer_features.loc[
    customer_features["marketing_segment"].eq("At-Risk High Value")
]
recent_one_time = customer_features.loc[
    customer_features["marketing_segment"].isin(
        ["New High Potential", "Recent One-Time Buyer"]
    )
]
frequent_low_aov = customer_features.loc[
    customer_features["marketing_segment"].eq("Frequent Low AOV")
]
champions_and_loyal = customer_features.loc[
    customer_features["marketing_segment"].isin(
        ["Champions", "Loyal High Value"]
    )
]
coupon_driven = customer_features.loc[
    customer_features["marketing_segment"].eq("Coupon Driven")
]

opportunity_rows = [
    {
        "opportunity": "Reactivate at-risk high-value customers",
        "target_customers": len(at_risk),
        "assumption": REACTIVATION_RATE,
        "estimated_incremental_net_sales": (
            len(at_risk)
            * REACTIVATION_RATE
            * at_risk["average_order_value"].median()
            if len(at_risk) else 0
        ),
    },
    {
        "opportunity": "Convert new and one-time buyers",
        "target_customers": len(recent_one_time),
        "assumption": SECOND_PURCHASE_CONVERSION_RATE,
        "estimated_incremental_net_sales": (
            len(recent_one_time)
            * SECOND_PURCHASE_CONVERSION_RATE
            * recent_one_time["average_order_value"].median()
            if len(recent_one_time) else 0
        ),
    },
    {
        "opportunity": "Increase frequent low-AOV baskets",
        "target_customers": len(frequent_low_aov),
        "assumption": CROSS_SELL_AOV_LIFT,
        "estimated_incremental_net_sales": (
            frequent_low_aov["monetary_value"].sum() * CROSS_SELL_AOV_LIFT
        ),
    },
    {
        "opportunity": "Expand champions and loyal customers",
        "target_customers": len(champions_and_loyal),
        "assumption": VIP_AOV_LIFT,
        "estimated_incremental_net_sales": (
            champions_and_loyal["monetary_value"].sum() * VIP_AOV_LIFT
        ),
    },
    {
        "opportunity": "Reduce coupon dependency",
        "target_customers": len(coupon_driven),
        "assumption": COUPON_REDUCTION_RATE,
        "estimated_incremental_net_sales": (
            coupon_driven["monetary_value"].sum()
            * coupon_driven["coupon_usage_rate"].mean()
            * COUPON_REDUCTION_RATE
            if len(coupon_driven) else 0
        ),
    },
]

opportunity_summary = pd.DataFrame(opportunity_rows)
opportunity_plot = opportunity_summary.sort_values("estimated_incremental_net_sales")

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(
    opportunity_plot["opportunity"],
    opportunity_plot["estimated_incremental_net_sales"],
    color=GOLD,
)
ax.set_title("Illustrative Incremental Net-Sales Opportunities")
ax.set_xlabel("Scenario estimate based on adjustable assumptions")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(currency_formatter)
ax.grid(axis="y", visible=False)
for bar in bars:
    ax.text(
        bar.get_width(), bar.get_y() + bar.get_height() / 2,
        f" ${bar.get_width():,.0f}", va="center", fontsize=9,
    )
save_figure(fig, "15_marketing_revenue_opportunity_scenarios.png")
display(opportunity_summary.sort_values("estimated_incremental_net_sales", ascending=False))

## Automated Marketing Insight Summary

In [ ]:
top_segment = segment_summary.nlargest(1, "historical_net_sales").iloc[0]
top_channel = (
    channel_quality.nlargest(1, "customer_net_sales").iloc[0]
    if not channel_quality.empty else None
)
top_product = (
    product_performance.nlargest(1, "net_sales").iloc[0]
    if not product_performance.empty else None
)
at_risk_value = at_risk["monetary_value"].sum()

insight_rows = [
    {
        "insight": "Revenue concentration",
        "result": f"{customer_share_for_80_percent:.1%} of identifiable customers generate 80% of completed customer net sales.",
        "action": "Protect this group with retention, VIP, and personalized service campaigns.",
    },
    {
        "insight": "Repeat purchasing",
        "result": f"{repeat_customer_rate:.1%} of identifiable customers have more than one completed order.",
        "action": "Use the second-purchase timing and cohort results to improve retention messaging.",
    },
    {
        "insight": "Largest customer segment by historical sales",
        "result": f"{top_segment['marketing_segment']} generated ${top_segment['historical_net_sales']:,.0f} in historical completed net sales.",
        "action": campaign_rules[top_segment["marketing_segment"]]["offer"],
    },
    {
        "insight": "At-risk historical value",
        "result": f"{len(at_risk):,} at-risk high-value customers previously generated ${at_risk_value:,.0f}.",
        "action": "Run a targeted win-back test and measure incremental reactivation against a holdout group.",
    },
    {
        "insight": "Customer identification coverage",
        "result": f"Customer IDs are available on {identifiable_customer_rate:.1%} of transaction records.",
        "action": "Improve customer capture so more sales can be connected to retention and lifetime-value analysis.",
    },
    {
        "insight": "Profit limitation",
        "result": "True margin, acquisition cost, and marketing spend are unavailable.",
        "action": "Add product cost and campaign spend before using this analysis for final profit or ROAS allocation.",
    },
]

if top_channel is not None:
    insight_rows.append(
        {
            "insight": "Top attribution source by customer sales",
            "result": f"{top_channel['primary_attribution']} generated ${top_channel['customer_net_sales']:,.0f} from identifiable customers.",
            "action": "Compare this source's marketing cost and incremental conversion before increasing spend.",
        }
    )

if top_product is not None:
    insight_rows.append(
        {
            "insight": "Top known product description",
            "result": f"{top_product['product_group']} generated ${top_product['net_sales']:,.0f} in completed net sales.",
            "action": "Use this product as a retention, bundle, and cross-sell anchor after validating missing product coverage.",
        }
    )

marketing_insights = pd.DataFrame(insight_rows)
marketing_insights.to_csv(INSIGHT_FILE, index=False)
display(marketing_insights)
print("Insight summary saved:", INSIGHT_FILE)

## Checks and recommended next actions

1. Start with small, measurable tests for At-Risk High Value, Recent One-Time Buyer, and Frequent Low AOV customers.
2. Use a randomly selected holdout group so incremental revenue can be separated from purchases that would have happened anyway.
3. Track campaign cost, discount amount, gross margin, delivery cost, and contribution profit.
4. Do not increase attribution-channel spend based only on historical sales; compare customer acquisition cost and incremental conversion.
5. Review the product parsing and missing-product rate before launching automated cross-sell recommendations.
6. Refresh customer segments regularly because recency and customer risk change over time.

### Additional columns needed for true profit analysis

- Product cost or cost of goods sold
- Discount amount
- Fulfilment and shipping cost
- Payment-processing fees
- Marketing spend by campaign and attribution source
- Campaign exposure, response, and control-group indicators

With these columns, true contribution profit can be calculated as:

`Contribution profit = Net sales - product cost - fulfilment cost - payment fees - marketing cost`